# PascalVOC-SP: Node-Representation Diagnostic & Pretrained Model Analysis

This notebook provides diagnostic evaluation and mechanistic analysis of HybridGraphFNet on PascalVOC-SP.

### Diagnostic Objectives:
1. **Trivial Majority-Class Baseline**: Measure Macro F1 and per-class F1 when predicting the dominant class (background) to quantify class imbalance impact.
2. **Evaluation of Pretrained Models (`models/`)**: Load and evaluate existing trained checkpoints (seeds 0, 1, 2) without needing 200 epochs of re-training.
3. **Per-Class Breakdown & Confusion Matrix**: Analyze whether errors concentrate in rare classes vs spatial smoothing across semantic boundaries.
4. **Controlled Diagnostic Ablations**: Screen feature dependencies (pixel statistics vs x/y coordinates vs Fourier positional encodings) and propagation mechanisms (local GCN vs global spectral vs raw node MLP).

## Sources & Reference
- LRGB repository: https://github.com/vijaydwivedi75/lrgb
- LRGB paper: https://arxiv.org/abs/2206.08164

In [ ]:
import os, io, csv, time, math, random, re, zipfile, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from tqdm.auto import tqdm
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix

from torch_geometric.datasets import LRGBDataset
from torch_geometric.loader import DataLoader
from torch_geometric.utils import to_dense_batch, to_dense_adj

warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# Multi-path resolution for root data, cache, and models
def resolve_path(candidates, default_path):
    for p in candidates:
        if os.path.exists(p):
            return p
    return default_path

DATA_ROOT = resolve_path(['./data', '../data', './new/data'], './data')
CACHE_DIR = resolve_path(['./eigenbasis_cache_voc_sp', '../eigenbasis_cache_voc_sp', './new/eigenbasis_cache_voc_sp'], './eigenbasis_cache_voc_sp')
MODEL_DIRS = [d for d in ['./models', '../models', '.', './new'] if os.path.exists(d)]

print('Data root:', DATA_ROOT)
print('Eigenbasis cache dir:', CACHE_DIR)
print('Model search dirs:', MODEL_DIRS)

TRUNC_K = 64
NUM_CLASSES = 21
CLASS_NAMES = [
    'background', 'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus',
    'car', 'cat', 'chair', 'cow', 'diningtable', 'dog', 'horse', 'motorbike',
    'person', 'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor'
]

HIDDEN_DIM = 128
NUM_LAYERS = 4
NUM_HEADS = 4
DROPOUT = 0.10
BATCH_SIZE = 4
ACCUM_STEPS = 4
LR = 1e-3
WEIGHT_DECAY = 1e-4

DIAGNOSTIC_SEEDS = [0]
MAX_EPOCHS = 60
PATIENCE = 12

RESULTS_CSV = 'diagnostic_ablation_results.csv'
MAIN_RESULTS_CSV = resolve_path(['results_pascalvoc_sp.csv', '../results_pascalvoc_sp.csv', './new/results_pascalvoc_sp.csv'], 'results_pascalvoc_sp.csv')

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


## 1. Load PascalVOC-SP Dataset & Verify Eigenbasis Cache

In [ ]:
train_ds = LRGBDataset(root=DATA_ROOT, name='PascalVOC-SP', split='train')
val_ds   = LRGBDataset(root=DATA_ROOT, name='PascalVOC-SP', split='val')
test_ds  = LRGBDataset(root=DATA_ROOT, name='PascalVOC-SP', split='test')

sample = train_ds[0]
NODE_FEAT_DIM = sample.x.shape[-1]
EDGE_DIM = sample.edge_attr.shape[-1] if (sample.edge_attr is not None and sample.edge_attr.dim() > 1) else 1

print('Graphs:', len(train_ds), 'train,', len(val_ds), 'val,', len(test_ds), 'test')
print('Node feature dim:', NODE_FEAT_DIM)
print('Edge feature dim:', EDGE_DIM)
print('Classes:', NUM_CLASSES)
print('Sample:', sample.num_nodes, 'nodes,', sample.num_edges, 'edges')

assert NODE_FEAT_DIM == 14, f'Expected 14-D PascalVOC-SP features, got {NODE_FEAT_DIM}.'

print('\nFeature breakdown:')
print('  0..11 : RGB pixel statistics (mean, std, min, max, etc.)')
print('  12..13: x/y center-of-mass spatial coordinates')

In [ ]:
class CachedEigenbasisDataset:
    def __init__(self, base, cache_dir, split, k_trunc=64):
        self.base = base
        self.split_dir = os.path.join(cache_dir, split)
        self.k_trunc = k_trunc

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        d = self.base[idx].clone()
        p = os.path.join(self.split_dir, f'{idx}.pt')
        if os.path.exists(p):
            cache = torch.load(p, weights_only=True)
            U = cache['U']
            if U.size(1) < self.k_trunc:
                U = F.pad(U, (0, self.k_trunc - U.size(1)))
            d.cached_U = U
        else:
            n = d.num_nodes
            d.cached_U = torch.eye(n)[:, :min(n, self.k_trunc)]
            if d.cached_U.size(1) < self.k_trunc:
                d.cached_U = F.pad(d.cached_U, (0, self.k_trunc - d.cached_U.size(1)))
        return d

def make_loaders(batch_size=BATCH_SIZE):
    return (
        DataLoader(CachedEigenbasisDataset(train_ds, CACHE_DIR, 'train', k_trunc=TRUNC_K),
                   batch_size=batch_size, shuffle=True, num_workers=0),
        DataLoader(CachedEigenbasisDataset(val_ds, CACHE_DIR, 'val', k_trunc=TRUNC_K),
                   batch_size=batch_size, shuffle=False, num_workers=0),
        DataLoader(CachedEigenbasisDataset(test_ds, CACHE_DIR, 'test', k_trunc=TRUNC_K),
                   batch_size=batch_size, shuffle=False, num_workers=0),
    )

for split in ['train', 'val', 'test']:
    p = Path(CACHE_DIR) / split
    print(split, 'cached eigenbases:', len(list(p.glob('*.pt'))) if p.exists() else 0)

## 2. Class Distribution, Majority-Class Baseline & Graph Homophily

PascalVOC-SP is evaluated via **Macro F1**, meaning all 21 classes contribute equally to the metric regardless of sample support.

We compute:
1. **Class Distribution**: Node counts and support percentages across train, validation, and test splits.
2. **Trivial Majority Baseline**: Macro F1 achieved by trivially predicting the dominant class (background) for all nodes.
3. **Spatial Graph Homophily**: Percentage of graph edges connecting superpixels that share identical semantic labels.

In [ ]:
def get_all_labels(ds):
    return torch.cat([d.y for d in ds], dim=0).numpy()

train_labels = get_all_labels(train_ds)
val_labels   = get_all_labels(val_ds)
test_labels  = get_all_labels(test_ds)

train_counts = np.bincount(train_labels, minlength=NUM_CLASSES)
val_counts   = np.bincount(val_labels, minlength=NUM_CLASSES)
test_counts  = np.bincount(test_labels, minlength=NUM_CLASSES)

majority_cls = int(np.argmax(train_counts))
majority_name = CLASS_NAMES[majority_cls]

print(f"Majority Class in Training Set: Class {majority_cls} ('{majority_name}')")
print(f"  Train Support: {train_counts[majority_cls]:,} / {len(train_labels):,} nodes ({100.0 * train_counts[majority_cls] / len(train_labels):.2f}%)")

# Trivial Majority Baseline Evaluation
def evaluate_majority_baseline(labels, maj_c):
    preds = np.full_like(labels, fill_value=maj_c)
    macro_f1 = f1_score(labels, preds, average='macro', zero_division=0)
    per_class_f1 = f1_score(labels, preds, labels=np.arange(NUM_CLASSES), average=None, zero_division=0)
    acc = accuracy_score(labels, preds)
    return macro_f1, per_class_f1, acc

val_maj_macro, val_maj_per_class, val_maj_acc = evaluate_majority_baseline(val_labels, majority_cls)
test_maj_macro, test_maj_per_class, test_maj_acc = evaluate_majority_baseline(test_labels, majority_cls)

print('\n=== TRIVIAL MAJORITY BASELINE ===')
print(f'Validation Set -> Macro F1: {val_maj_macro:.4f} | Accuracy: {val_maj_acc:.4f}')
print(f'Test Set       -> Macro F1: {test_maj_macro:.4f} | Accuracy: {test_maj_acc:.4f}')
print(f'(Note: Class {majority_cls} F1 = {test_maj_per_class[majority_cls]:.4f}, all other 20 classes = 0.0000)')

class_table = pd.DataFrame({
    'id': np.arange(NUM_CLASSES),
    'class_name': CLASS_NAMES,
    'train_support': train_counts,
    'train_pct': 100.0 * train_counts / len(train_labels),
    'test_support': test_counts,
    'test_pct': 100.0 * test_counts / len(test_labels),
    'maj_baseline_f1': test_maj_per_class,
})

print('\n--- Top 10 Most Common Classes ---')
print(class_table.sort_values('train_support', ascending=False).head(10).to_string(index=False))
print('\n--- Top 10 Rarest Classes ---')
print(class_table.sort_values('train_support', ascending=True).head(10).to_string(index=False))

In [ ]:
def graph_locality(ds, max_graphs=1000):
    rng = np.random.default_rng(0)
    idxs = np.arange(len(ds)) if len(ds) <= max_graphs else rng.choice(len(ds), max_graphs, replace=False)
    same = total = majority = n_nodes = 0
    ent = []

    for idx in tqdm(idxs, desc='locality', leave=False):
        d = ds[int(idx)]
        y = d.y.cpu()
        src, dst = d.edge_index.cpu()

        if src.numel():
            same += int((y[src] == y[dst]).sum())
            total += int(src.numel())

        neigh = [[] for _ in range(d.num_nodes)]
        for s, t in zip(src.tolist(), dst.tolist()):
            if s != t:
                neigh[s].append(t)
                neigh[t].append(s)

        for i, ns in enumerate(neigh):
            if not ns:
                continue
            labels = y[torch.tensor(ns)]
            vals, counts = torch.unique(labels, return_counts=True)
            majority += int(vals[counts.argmax()] == y[i])
            n_nodes += 1
            p = counts.float() / counts.sum()
            ent.append(float(-(p * torch.log(p + 1e-12)).sum()))

    return {
        'same_label_edge_fraction': same / max(total, 1),
        'neighbor_majority_accuracy': majority / max(n_nodes, 1),
        'mean_neighbor_label_entropy': float(np.mean(ent)),
        'graphs_sampled': len(idxs),
    }

locality = graph_locality(test_ds)
print('\n--- Graph Homophily & Locality Diagnostics ---')
for k, v in locality.items():
    print(f'  {k}: {v}')

## 3. Architecture & DiagnosticGraphFNet

The architecture reproduces the node-level HybridGraphFNet with flexible switches for feature and branch ablations.

- `self.node_classifier`: Per-node classification head matching pretrained checkpoints.
- `edge_dim=EDGE_DIM`: Edge MLP matching 2-D edge attributes in PascalVOC-SP.

In [ ]:
class DenseGCNLayer(nn.Module):
    def __init__(self, hidden_dim, edge_dim=EDGE_DIM):
        super().__init__()
        self.node_lin = nn.Linear(hidden_dim, hidden_dim)
        self.edge_lin = nn.Linear(edge_dim, 1)
        self.norm     = nn.LayerNorm(hidden_dim)

    def forward(self, x, A_norm, edge_dense=None):
        if edge_dense is not None:
            e = torch.sigmoid(self.edge_lin(edge_dense)).squeeze(-1)
            msg = torch.bmm(A_norm * e, x)
        else:
            msg = torch.bmm(A_norm, x)
        return self.norm(F.gelu(self.node_lin(msg)))

class SpectralMix(nn.Module):
    def __init__(self, hidden_dim, num_heads=4):
        super().__init__()
        self.num_heads  = num_heads
        self.head_dim   = hidden_dim // num_heads
        self.filter_gen = nn.Linear(hidden_dim, hidden_dim)
        self.out_proj   = nn.Linear(hidden_dim, hidden_dim)
        self.norm       = nn.LayerNorm(hidden_dim)

    def forward(self, x, U, mask):
        x_hat      = torch.bmm(U.transpose(1, 2), x)
        fil        = torch.sigmoid(self.filter_gen(x_hat))
        x_filtered = fil * x_hat
        x_out      = torch.bmm(U, x_filtered)
        x_out      = x_out * mask.unsqueeze(-1)
        return self.norm(self.out_proj(F.gelu(x_out)))

def fourier_coords(x, freqs=5):
    c = x[..., 12:14]
    mu = c.mean(1, keepdim=True)
    sd = c.std(1, keepdim=True).clamp_min(1e-6)
    z = (c - mu) / sd
    out = [x]
    for i in range(freqs):
        w = (2.0**i) * math.pi
        out += [torch.sin(w * z), torch.cos(w * z)]
    return torch.cat(out, dim=-1)

class DiagnosticGraphFNet(nn.Module):
    def __init__(self, feature_mode='full', branch_mode='full', in_dim=NODE_FEAT_DIM, hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS, num_classes=NUM_CLASSES, edge_dim=EDGE_DIM, lap_k=8, dropout=DROPOUT):
        super().__init__()
        self.feature_mode = feature_mode
        self.branch_mode = branch_mode
        self.lap_k = lap_k
        inp = 34 if feature_mode == 'fourier_coord' else in_dim

        self.input_proj = nn.Sequential(
            nn.Linear(inp, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, hidden_dim)
        )
        self.pe_encoder = nn.Linear(lap_k, hidden_dim)

        self.layers = nn.ModuleList([
            nn.ModuleDict({
                'local':  DenseGCNLayer(hidden_dim, edge_dim=edge_dim),
                'global': SpectralMix(hidden_dim),
                'gate':   nn.Linear(hidden_dim, hidden_dim),
                'norm':   nn.LayerNorm(hidden_dim),
            }) for _ in range(num_layers)
        ])

        self.node_classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(hidden_dim, num_classes)
        )

    @property
    def head(self):
        return self.node_classifier

    def transform_x(self, x):
        if self.feature_mode == 'full':
            return x
        if self.feature_mode == 'no_pixel':
            z = x.clone(); z[..., :12] = 0; return z
        if self.feature_mode == 'no_coord':
            z = x.clone(); z[..., 12:14] = 0; return z
        if self.feature_mode == 'pixel_only':
            z = x.clone(); z[..., 12:14] = 0; return z
        if self.feature_mode == 'coord_only':
            z = x.clone(); z[..., :12] = 0; return z
        if self.feature_mode == 'no_raw_features':
            return torch.zeros_like(x)
        if self.feature_mode == 'fourier_coord':
            return fourier_coords(x)
        raise ValueError(f'Unknown feature_mode: {self.feature_mode}')

    @staticmethod
    def compute_A_norm(adj, mask):
        B, N, _ = adj.shape
        out = []
        for b in range(B):
            n = int(mask[b].sum())
            a = adj[b, :n, :n]
            d = a.sum(1).clamp_min(1e-8).rsqrt()
            out.append(F.pad(d[:, None] * a * d[None, :], (0, N - n, 0, N - n)))
        return torch.stack(out)

    def forward(self, data):
        x, mask = to_dense_batch(data.x.float(), data.batch)
        adj = to_dense_adj(data.edge_index, data.batch, max_num_nodes=x.size(1))
        if data.edge_attr is not None:
            ea = data.edge_attr.float()
            if ea.dim() == 1: ea = ea[:, None]
            edge_dense = to_dense_adj(data.edge_index, data.batch, edge_attr=ea, max_num_nodes=x.size(1))
        else:
            edge_dense = None

        adj = adj + torch.eye(adj.size(1), device=x.device)[None]
        A = self.compute_A_norm(adj, mask)

        U, _ = to_dense_batch(data.cached_U.float(), data.batch)
        U = U * mask.unsqueeze(-1)

        x = self.input_proj(self.transform_x(x))
        k = min(self.lap_k, U.size(-1))
        lap_pe = U[:, :, :k] * mask.unsqueeze(-1)
        x = x + self.pe_encoder(lap_pe)

        for layer in self.layers:
            res = x
            if self.branch_mode == 'mlp_only':
                x = layer['norm'](res)
                continue

            xl = layer['local'](x, A, edge_dense)
            xg = layer['global'](x, U, mask)

            if self.branch_mode == 'local_only':
                xm = xl
            elif self.branch_mode == 'global_only':
                xm = xg
            elif self.branch_mode == 'no_graph':
                xm = torch.zeros_like(x)
            else:
                g = torch.sigmoid(layer['gate'](x))
                xm = g * xl + (1 - g) * xg

            x = layer['norm'](res + F.dropout(xm, p=DROPOUT, training=self.training))

        return self.node_classifier(x * mask.unsqueeze(-1)), mask

print('DiagnosticGraphFNet defined.')

## 4. Evaluation of Pretrained Initial Run Checkpoints (`models/`)

Here we load the trained checkpoints `best_model_voc_sp_seed0`, `best_model_voc_sp_seed1`, `best_model_voc_sp_seed2` from `models/`.

We evaluate each checkpoint on validation and test sets to obtain:
- Macro F1, Micro F1, Weighted F1, and Accuracy
- Per-Class F1 breakdown table
- Normalized confusion matrices
- Comparison against the Trivial Majority Baseline

In [ ]:
def load_checkpoint_dict(ckpt_path, target_device=device):
    """Loads PyTorch state_dict from .pt file, .zip archive, or unzipped directory."""
    if os.path.isdir(ckpt_path):
        buffer = io.BytesIO()
        prefix = os.path.basename(ckpt_path)
        with zipfile.ZipFile(buffer, 'w', zipfile.ZIP_STORED) as zf:
            for root, dirs, files in os.walk(ckpt_path):
                for f in files:
                    if f.startswith('.'): continue
                    full_p = os.path.join(root, f)
                    rel_p = os.path.join(prefix, os.path.relpath(full_p, ckpt_path))
                    zinfo = zipfile.ZipInfo(filename=rel_p, date_time=(2024, 1, 1, 0, 0, 0))
                    with open(full_p, 'rb') as fp:
                        zf.writestr(zinfo, fp.read())
        buffer.seek(0)
        state = torch.load(buffer, map_location=target_device, weights_only=True)
    else:
        try:
            state = torch.load(ckpt_path, map_location=target_device, weights_only=True)
        except Exception:
            state = torch.load(ckpt_path, map_location=target_device, weights_only=False)
    if isinstance(state, dict) and 'model_state' in state:
        state = state['model_state']
    return state

def evaluate(model, loader):
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for b in tqdm(loader, desc='eval', leave=False):
            b = b.to(device)
            logits, mask = model(b)
            y, _ = to_dense_batch(b.y, b.batch, fill_value=-1)
            valid = mask.bool()
            ps.append(logits[valid].argmax(-1).cpu())
            ys.append(y[valid].cpu())

    y_true = torch.cat(ys).numpy()
    y_pred = torch.cat(ps).numpy()

    return {
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'micro_f1': f1_score(y_true, y_pred, average='micro', zero_division=0),
        'weighted_f1': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'accuracy': accuracy_score(y_true, y_pred),
        'per_class_f1': f1_score(y_true, y_pred, labels=np.arange(NUM_CLASSES), average=None, zero_division=0),
        'y_true': y_true,
        'y_pred': y_pred,
        'cm': confusion_matrix(y_true, y_pred, labels=np.arange(NUM_CLASSES)),
    }

# Discover all saved PascalVOC checkpoints in models/ and current directory
found_ckpts = []
for d in MODEL_DIRS:
    for f in os.listdir(d):
        if 'best_model_voc_sp_seed' in f:
            full_path = os.path.join(d, f)
            found_ckpts.append(full_path)

found_ckpts = sorted(list(set(found_ckpts)))
print(f'Discovered {len(found_ckpts)} PascalVOC-SP checkpoints: {[os.path.basename(p) for p in found_ckpts]}')

train_loader, val_loader, test_loader = make_loaders(batch_size=8)
seed_eval_results = {}

for ckpt_path in found_ckpts:
    fname = os.path.basename(ckpt_path)
    seed_match = re.search(r'seed(\d+)', fname)
    seed_id = int(seed_match.group(1)) if seed_match else fname

    print(f'\nEvaluating checkpoint: {fname} (Seed {seed_id})...')
    m = DiagnosticGraphFNet(feature_mode='full', branch_mode='full').to(device)
    state_dict = load_checkpoint_dict(ckpt_path, device)
    m.load_state_dict(state_dict, strict=True)

    val_metrics = evaluate(m, val_loader)
    test_metrics = evaluate(m, test_loader)

    seed_eval_results[seed_id] = {
        'val': val_metrics,
        'test': test_metrics,
        'path': ckpt_path,
    }
    print(f"  Val Macro F1:  {val_metrics['macro_f1']:.4f} | Accuracy: {val_metrics['accuracy']:.4f}")
    print(f"  Test Macro F1: {test_metrics['macro_f1']:.4f} | Accuracy: {test_metrics['accuracy']:.4f}")

In [ ]:
if seed_eval_results:
    seeds = sorted(list(seed_eval_results.keys()))
    total_test_nodes = len(test_labels)

    print('=' * 95)
    print('PER-CLASS F1 BREAKDOWN TABLE (TEST SET) VS MAJORITY BASELINE')
    print('=' * 95)

    header = f"{'ID':<3} {'Class Name':<14} {'Support':<9} {'% Nodes':<8} {'Maj Base':<10}"
    for s in seeds:
        header += f" {'Seed '+str(s):<10}"
    if len(seeds) > 1:
        header += f" {'Mean F1':<10}"
    print(header)
    print('-' * len(header))

    for c in range(NUM_CLASSES):
        sup = test_counts[c]
        pct = 100.0 * sup / total_test_nodes
        maj_f1 = test_maj_per_class[c]
        row = f"{c:<3d} {CLASS_NAMES[c]:<14} {sup:<9,d} {pct:<7.2f}% {maj_f1:<10.4f}"
        c_f1s = []
        for s in seeds:
            f1_s = seed_eval_results[s]['test']['per_class_f1'][c]
            c_f1s.append(f1_s)
            row += f" {f1_s:<10.4f}"
        if len(seeds) > 1:
            row += f" {np.mean(c_f1s):<10.4f}"
        print(row)

    print('-' * len(header))
    summary_row = f"{'MACRO F1 AVERAGE':<28} {total_test_nodes:<9,d} 100.00% {test_maj_macro:<10.4f}"
    all_macros = []
    for s in seeds:
        m_s = seed_eval_results[s]['test']['macro_f1']
        all_macros.append(m_s)
        summary_row += f" {m_s:<10.4f}"
    if len(seeds) > 1:
        summary_row += f" {np.mean(all_macros):<10.4f}"
    print(summary_row)
    print('=' * 95)

    # Visualization: Support vs Per-Class F1
    rep_seed = seeds[0]
    rep_res = seed_eval_results[rep_seed]['test']

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    ax1.scatter(test_counts.clip(min=1), rep_res['per_class_f1'], s=80, alpha=0.85, c='royalblue', edgecolors='black')
    for i, txt in enumerate(CLASS_NAMES):
        ax1.annotate(txt, (test_counts[i], rep_res['per_class_f1'][i]), fontsize=8, alpha=0.75)
    ax1.set_xscale('log')
    ax1.set_xlabel('Test Set Node Support (log scale)')
    ax1.set_ylabel('Test Per-Class F1')
    ax1.set_title(f'Class Support vs F1 Score (Seed {rep_seed})')
    ax1.grid(True, alpha=0.3)

    # Normalized Confusion Matrix
    cm = rep_res['cm']
    cmn = cm.astype('float') / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    im = ax2.imshow(cmn, aspect='auto', cmap='Blues')
    ax2.set_xlabel('Predicted Class')
    ax2.set_ylabel('True Class')
    ax2.set_title(f'Row-Normalized Confusion Matrix (Seed {rep_seed})')
    fig.colorbar(im, ax=ax2)

    plt.tight_layout()
    plt.show()
else:
    print('No pretrained checkpoints found to display.')

## 5. Controlled Diagnostic Ablations Matrix

We define hypothesis-driven ablation configurations to isolate:
1. **Feature reliance**: `full`, `no_pixel`, `no_coord`, `pixel_only`, `coord_only`, `fourier_coord`
2. **Propagation mechanics**: `full`, `local_only`, `global_only`, `no_graph`, `mlp_only`

In [ ]:
ABLATIONS = [
    ('full', 'full', 'full'),
    ('no_pixel', 'no_pixel', 'full'),
    ('no_coord', 'no_coord', 'full'),
    ('pixel_only', 'pixel_only', 'full'),
    ('coord_only', 'coord_only', 'full'),
    ('local_only', 'full', 'local_only'),
    ('global_only', 'full', 'global_only'),
    ('no_graph', 'full', 'no_graph'),
    ('mlp_only', 'full', 'mlp_only'),
    ('fourier_coord', 'fourier_coord', 'full'),
]

pd.DataFrame(ABLATIONS, columns=['name', 'feature_mode', 'branch_mode'])

## 6. Ablation Runner (Screening & Checkpoint Resumption)

The screening runner saves the best checkpoint for each ablation to `diag_best_<name>_seed<seed>.pt` and full resumption state to `diag_resume_<name>_seed<seed>.pt`.

In [ ]:
def compute_class_weights(dataset):
    y = torch.cat([d.y for d in dataset], dim=0)
    c = torch.bincount(y, minlength=NUM_CLASSES).float().clamp_min(1)
    w = 1.0 / c
    return (w / w.sum() * NUM_CLASSES).to(device)

CW = compute_class_weights(train_ds)

def save_resume(path, model, opt, sched, epoch, best_val, best_epoch, no_improve):
    torch.save({
        'model_state': model.state_dict(),
        'optimizer_state': opt.state_dict(),
        'scheduler_state': sched.state_dict(),
        'epoch': epoch,
        'best_val': best_val,
        'best_epoch': best_epoch,
        'no_improve': no_improve,
    }, path)

def train_one(name, feature_mode, branch_mode, seed):
    set_seed(seed)
    train_loader, val_loader, test_loader = make_loaders(batch_size=BATCH_SIZE)

    model = DiagnosticGraphFNet(feature_mode=feature_mode, branch_mode=branch_mode).to(device)
    opt = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MAX_EPOCHS, eta_min=1e-5)
    crit = nn.CrossEntropyLoss(weight=CW, ignore_index=-1)

    resume = f'diag_resume_{name}_seed{seed}.pt'
    bestp  = f'diag_best_{name}_seed{seed}.pt'

    start = 1; best_val = 0; best_epoch = 0; no_improve = 0
    if os.path.exists(resume):
        ck = torch.load(resume, map_location=device, weights_only=False)
        model.load_state_dict(ck['model_state'])
        opt.load_state_dict(ck['optimizer_state'])
        sched.load_state_dict(ck['scheduler_state'])
        start = ck['epoch'] + 1
        best_val = ck['best_val']
        best_epoch = ck['best_epoch']
        no_improve = ck['no_improve']
        print(f'Resuming {name} (seed {seed}) from epoch {start}')

    t0 = time.time()

    for ep in range(start, MAX_EPOCHS + 1):
        model.train()
        opt.zero_grad(set_to_none=True)
        running = 0

        for step, b in tqdm(enumerate(train_loader), total=len(train_loader), desc=f'{name} | seed {seed} | ep {ep}', leave=False):
            b = b.to(device)
            logits, mask = model(b)
            y, _ = to_dense_batch(b.y, b.batch, fill_value=-1)
            loss = crit(logits.reshape(-1, logits.size(-1)), y.reshape(-1).long()) / ACCUM_STEPS
            loss.backward()
            running += loss.item() * ACCUM_STEPS

            if (step + 1) % ACCUM_STEPS == 0 or step == len(train_loader) - 1:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
                opt.zero_grad(set_to_none=True)

        sched.step()
        vm = evaluate(model, val_loader)['macro_f1']

        if vm > best_val:
            best_val = vm; best_epoch = ep; no_improve = 0
            torch.save(model.state_dict(), bestp)
        else:
            no_improve += 1

        if ep % 5 == 0 or no_improve == 0:
            print(f'ep {ep:3d} | loss {running/len(train_loader):.4f} | val F1 {vm:.4f} | best {best_val:.4f}@ep{best_epoch}')

        if ep % 5 == 0:
            save_resume(resume, model, opt, sched, ep, best_val, best_epoch, no_improve)

        if no_improve >= PATIENCE:
            print(f'Early stopping at epoch {ep} (patience={PATIENCE})')
            break

    if os.path.exists(bestp):
        model.load_state_dict(torch.load(bestp, map_location=device, weights_only=True))
    tm = evaluate(model, test_loader)

    row = {
        'ablation': name,
        'feature_mode': feature_mode,
        'branch_mode': branch_mode,
        'seed': seed,
        'test_macro_f1': tm['macro_f1'],
        'test_micro_f1': tm['micro_f1'],
        'test_weighted_f1': tm['weighted_f1'],
        'test_accuracy': tm['accuracy'],
        'best_epoch': best_epoch,
        'time_s': time.time() - t0,
        'params': count_params(model),
    }
    print('DONE:', row)
    return row, model, tm

In [ ]:
# Set RUN_SCREENING = True if you wish to run/resume the training ablation experiments
RUN_SCREENING = False
rows = []

if RUN_SCREENING:
    for name, feature_mode, branch_mode in ABLATIONS:
        for seed in DIAGNOSTIC_SEEDS:
            try:
                row, model, tm = train_one(name, feature_mode, branch_mode, seed)
                rows.append(row)
                pd.DataFrame(rows).to_csv(RESULTS_CSV, index=False)
            except Exception as e:
                print(f'FAILED {name} (seed {seed}): {type(e).__name__}: {e}')

results_df = pd.DataFrame(rows)
if not len(results_df) and os.path.exists(RESULTS_CSV):
    results_df = pd.read_csv(RESULTS_CSV)

if len(results_df):
    print(results_df.sort_values('test_macro_f1', ascending=False).to_string(index=False))
else:
    print('No ablation CSV found. Evaluation using pretrained models from models/ is available in Section 4 above.')

## 7. Comparative Analysis & Diagnostic Conclusions

In [ ]:
print('=' * 80)
print('PASCALVOC-SP DIAGNOSTIC SUMMARY & FINDINGS')
print('=' * 80)

print('1. TRIVIAL MAJORITY-CLASS BASELINE:')
print(f'   Dominant Class:        Class {majority_cls} (\'{majority_name}\')')
print(f'   Dominant Node Support: {100.0 * test_counts[majority_cls] / len(test_labels):.2f}% of all test nodes')
print(f'   Baseline Test Macro F1:{test_maj_macro:.4f} (Accuracy: {test_maj_acc:.4f})')

if seed_eval_results:
    all_macros = [seed_eval_results[s]['test']['macro_f1'] for s in seed_eval_results]
    all_accs   = [seed_eval_results[s]['test']['accuracy'] for s in seed_eval_results]
    print('\n2. PRETRAINED HYBRIDGRAPHFNET PERFORMANCE:')
    print(f'   Mean Test Macro F1:    {np.mean(all_macros):.4f} +/- {np.std(all_macros):.4f}')
    print(f'   Mean Test Accuracy:    {np.mean(all_accs):.4f} +/- {np.std(all_accs):.4f}')
    print(f'   Gain over Majority:    {np.mean(all_macros) - test_maj_macro:+.4f} Macro F1 points')

    rep_seed = sorted(list(seed_eval_results.keys()))[0]
    rep_f1s = seed_eval_results[rep_seed]['test']['per_class_f1']
    bg_f1 = rep_f1s[majority_cls]
    non_bg_f1s = [rep_f1s[i] for i in range(NUM_CLASSES) if i != majority_cls]
    zero_f1_count = sum(1 for f in non_bg_f1s if f < 0.01)

    print('\n3. PER-CLASS BOTTLENECK ANALYSIS:')
    print(f'   Background F1:         {bg_f1:.4f}')
    print(f'   Non-Background Mean F1:{np.mean(non_bg_f1s):.4f}')
    print(f'   Classes with F1 < 0.01:{zero_f1_count} / {NUM_CLASSES - 1} non-background classes')

print('\n4. GRAPH HOMOPHILY:')
print(f'   Same-label edge fraction: {locality["same_label_edge_fraction"]:.4f}')
print(f'   Neighbor majority acc:    {locality["neighbor_majority_accuracy"]:.4f}')
print('=' * 80)
